# NYC Taxi Gold Analysis

Este notebook responde las **20 preguntas de negocio** usando **solo tablas `gold.*`**, como exige el PSet.  
Las preguntas y el requisito de usar únicamente la capa Gold están definidos en el enunciado. fileciteturn4file0

## Supuestos de esquema Gold
Este notebook asume que existen al menos estas tablas:

- `gold.fct_trips`
- `gold.dim_date`
- `gold.dim_zone`
- `gold.dim_service_type`
- `gold.dim_payment_type`

Y que `gold.fct_trips` contiene, como mínimo, llaves hacia fecha, zona de pickup, zona de dropoff, tipo de servicio y tipo de pago, además de métricas como duración, distancia, tarifa, propina y total.

> Si alguno de tus nombres finales cambió, ajusta los SQL en las celdas correspondientes.


In [ ]:

import os
from textwrap import shorten

import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'localhost')
POSTGRES_PORT = int(os.getenv('POSTGRES_PORT', '5432'))
POSTGRES_DB = os.getenv('POSTGRES_DB', 'nyc_taxi')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')

engine = create_engine(
    f'postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}'
)

print(f'Connected config -> host={POSTGRES_HOST} port={POSTGRES_PORT} db={POSTGRES_DB} user={POSTGRES_USER}')


In [ ]:

def run_sql(sql: str, limit: int | None = None) -> pd.DataFrame:
    sql_to_run = sql
    if limit is not None:
        sql_to_run = f"SELECT * FROM ({sql}) AS q LIMIT {limit}"
    return pd.read_sql(text(sql_to_run), engine)

def show_question(n: int, title: str, tables: str):
    display(Markdown(f"## {n}. {title}\n**Tablas usadas:** `{tables}`"))

def show_sql(sql: str):
    display(Markdown("**SQL usado:**"))
    print(sql)

def basic_interpretation(df: pd.DataFrame, value_col: str | None = None, label: str = "resultado") -> None:
    if df.empty:
        display(Markdown(f"**Interpretación:** No se encontraron datos para {label}."))        
        return
    if value_col and value_col in df.columns:
        top = df.iloc[0]
        display(Markdown(
            f"**Interpretación:** El valor más alto observado en este resultado es **{top[value_col]}**. "
            f"Revisa la primera fila para identificar la dimensión asociada y redactar tu conclusión final de 1–3 líneas."
        ))
    else:
        display(Markdown(
            "**Interpretación:** Revisa el resultado y redacta una conclusión de 1–3 líneas sobre el patrón más importante."
        ))


In [ ]:

required_tables = [
    'gold.fct_trips',
    'gold.dim_date',
    'gold.dim_zone',
    'gold.dim_service_type',
    'gold.dim_payment_type',
]

check_sql = '''
SELECT table_schema || '.' || table_name AS table_name
FROM information_schema.tables
WHERE table_schema = 'gold'
ORDER BY table_name;
'''

gold_tables = run_sql(check_sql)
display(gold_tables)
missing = sorted(set(required_tables) - set(gold_tables['table_name'].tolist()))
if missing:
    print('WARNING: faltan estas tablas esperadas:', missing)
else:
    print('OK: se encontraron las tablas gold esperadas.')


### Pregunta 1
**Viajes por mes (2024): count(*) por month.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(1, 'Viajes por mes (2024): count(*) por month.', 'gold.fct_trips, gold.dim_date')
sql = """SELECT
    d.year,
    d.month,
    COUNT(*) AS trip_count
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE d.year = 2024
GROUP BY d.year, d.month
ORDER BY d.year, d.month"""
show_sql(sql)
df_1 = run_sql(sql)
display(df_1)
basic_interpretation(df_1, value_col='trip_count', label='Viajes por mes (2024): count(*) por month.')


### Pregunta 2
**Viajes por service_type y mes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(2, 'Viajes por service_type y mes.', 'gold.fct_trips, gold.dim_date, gold.dim_service_type')
sql = """SELECT
    d.year,
    d.month,
    st.service_type,
    COUNT(*) AS trip_count
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_service_type st
    ON f.service_type_key = st.service_type_key
GROUP BY d.year, d.month, st.service_type
ORDER BY d.year, d.month, st.service_type"""
show_sql(sql)
df_2 = run_sql(sql)
display(df_2)
basic_interpretation(df_2, value_col='trip_count', label='Viajes por service_type y mes.')


### Pregunta 3
**Top 10 zonas de pickup (total 2024).**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(3, 'Top 10 zonas de pickup (total 2024).', 'gold.fct_trips, gold.dim_date, gold.dim_zone')
sql = """SELECT
    z.zone AS pickup_zone,
    z.borough AS pickup_borough,
    COUNT(*) AS trip_count
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_zone z
    ON f.pickup_zone_id = z.zone_id
WHERE d.year = 2024
GROUP BY z.zone, z.borough
ORDER BY trip_count DESC
LIMIT 10"""
show_sql(sql)
df_3 = run_sql(sql)
display(df_3)
basic_interpretation(df_3, value_col='trip_count', label='Top 10 zonas de pickup (total 2024).')


### Pregunta 4
**Top 10 zonas de dropoff (total 2024).**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(4, 'Top 10 zonas de dropoff (total 2024).', 'gold.fct_trips, gold.dim_date, gold.dim_zone')
sql = """SELECT
    z.zone AS dropoff_zone,
    z.borough AS dropoff_borough,
    COUNT(*) AS trip_count
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_zone z
    ON f.dropoff_zone_id = z.zone_id
WHERE d.year = 2024
GROUP BY z.zone, z.borough
ORDER BY trip_count DESC
LIMIT 10"""
show_sql(sql)
df_4 = run_sql(sql)
display(df_4)
basic_interpretation(df_4, value_col='trip_count', label='Top 10 zonas de dropoff (total 2024).')


### Pregunta 5
**Top 5 boroughs por mes (pickup).**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(5, 'Top 5 boroughs por mes (pickup).', 'gold.fct_trips, gold.dim_date, gold.dim_zone')
sql = """WITH ranked AS (
    SELECT
        d.year,
        d.month,
        z.borough,
        COUNT(*) AS trip_count,
        ROW_NUMBER() OVER (
            PARTITION BY d.year, d.month
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM gold.fct_trips f
    JOIN gold.dim_date d
        ON f.pickup_date_key = d.date_key
    JOIN gold.dim_zone z
        ON f.pickup_zone_id = z.zone_id
    GROUP BY d.year, d.month, z.borough
)
SELECT year, month, borough, trip_count
FROM ranked
WHERE rn <= 5
ORDER BY year, month, trip_count DESC"""
show_sql(sql)
df_5 = run_sql(sql)
display(df_5)
basic_interpretation(df_5, value_col='trip_count', label='Top 5 boroughs por mes (pickup).')


### Pregunta 6
**Horas pico (top 5 horas) para cada día de semana.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(6, 'Horas pico (top 5 horas) para cada día de semana.', 'gold.fct_trips, gold.dim_date')
sql = """WITH hourly AS (
    SELECT
        d.weekday_name,
        d.day_of_week,
        EXTRACT(HOUR FROM f.pickup_datetime) AS pickup_hour,
        COUNT(*) AS trip_count,
        ROW_NUMBER() OVER (
            PARTITION BY d.day_of_week
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM gold.fct_trips f
    JOIN gold.dim_date d
        ON f.pickup_date_key = d.date_key
    GROUP BY d.weekday_name, d.day_of_week, EXTRACT(HOUR FROM f.pickup_datetime)
)
SELECT weekday_name, day_of_week, pickup_hour, trip_count
FROM hourly
WHERE rn <= 5
ORDER BY day_of_week, trip_count DESC"""
show_sql(sql)
df_6 = run_sql(sql)
display(df_6)
basic_interpretation(df_6, value_col='trip_count', label='Horas pico (top 5 horas) para cada día de semana.')


### Pregunta 7
**Distribución de viajes por día de semana (ranking).**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(7, 'Distribución de viajes por día de semana (ranking).', 'gold.fct_trips, gold.dim_date')
sql = """SELECT
    d.weekday_name,
    d.day_of_week,
    COUNT(*) AS trip_count,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS trip_rank
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
GROUP BY d.weekday_name, d.day_of_week
ORDER BY trip_rank, d.day_of_week"""
show_sql(sql)
df_7 = run_sql(sql)
display(df_7)
basic_interpretation(df_7, value_col='trip_count', label='Distribución de viajes por día de semana (ranking).')


### Pregunta 8
**Ingreso total (total_amount) por mes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(8, 'Ingreso total (total_amount) por mes.', 'gold.fct_trips, gold.dim_date')
sql = """SELECT
    d.year,
    d.month,
    ROUND(SUM(f.total_amount)::numeric, 2) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month"""
show_sql(sql)
df_8 = run_sql(sql)
display(df_8)
basic_interpretation(df_8, value_col='total_revenue', label='Ingreso total (total_amount) por mes.')


### Pregunta 9
**Ingreso total por service_type y mes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(9, 'Ingreso total por service_type y mes.', 'gold.fct_trips, gold.dim_date, gold.dim_service_type')
sql = """SELECT
    d.year,
    d.month,
    st.service_type,
    ROUND(SUM(f.total_amount)::numeric, 2) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_service_type st
    ON f.service_type_key = st.service_type_key
GROUP BY d.year, d.month, st.service_type
ORDER BY d.year, d.month, st.service_type"""
show_sql(sql)
df_9 = run_sql(sql)
display(df_9)
basic_interpretation(df_9, value_col='total_revenue', label='Ingreso total por service_type y mes.')


### Pregunta 10
**tip % promedio por mes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(10, 'tip % promedio por mes.', 'gold.fct_trips, gold.dim_date')
sql = """SELECT
    d.year,
    d.month,
    ROUND(AVG(f.tip_amount / NULLIF(f.fare_amount, 0))::numeric, 4) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
WHERE f.fare_amount > 0
GROUP BY d.year, d.month
ORDER BY d.year, d.month"""
show_sql(sql)
df_10 = run_sql(sql)
display(df_10)
basic_interpretation(df_10, value_col='avg_tip_pct', label='tip % promedio por mes.')


### Pregunta 11
**tip % por borough y mes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(11, 'tip % por borough y mes.', 'gold.fct_trips, gold.dim_date, gold.dim_zone')
sql = """SELECT
    d.year,
    d.month,
    z.borough,
    ROUND(AVG(f.tip_amount / NULLIF(f.fare_amount, 0))::numeric, 4) AS avg_tip_pct
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
JOIN gold.dim_zone z
    ON f.pickup_zone_id = z.zone_id
WHERE f.fare_amount > 0
GROUP BY d.year, d.month, z.borough
ORDER BY d.year, d.month, avg_tip_pct DESC"""
show_sql(sql)
df_11 = run_sql(sql)
display(df_11)
basic_interpretation(df_11, value_col='avg_tip_pct', label='tip % por borough y mes.')


### Pregunta 12
**Top 10 zonas por ingreso total (pickup).**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(12, 'Top 10 zonas por ingreso total (pickup).', 'gold.fct_trips, gold.dim_zone')
sql = """SELECT
    z.zone AS pickup_zone,
    z.borough AS pickup_borough,
    ROUND(SUM(f.total_amount)::numeric, 2) AS total_revenue
FROM gold.fct_trips f
JOIN gold.dim_zone z
    ON f.pickup_zone_id = z.zone_id
GROUP BY z.zone, z.borough
ORDER BY total_revenue DESC
LIMIT 10"""
show_sql(sql)
df_12 = run_sql(sql)
display(df_12)
basic_interpretation(df_12, value_col='total_revenue', label='Top 10 zonas por ingreso total (pickup).')


### Pregunta 13
**Top 10 zonas por tip % (pickup) con mínimo N viajes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(13, 'Top 10 zonas por tip % (pickup) con mínimo N viajes.', 'gold.fct_trips, gold.dim_zone')
sql = """WITH zone_stats AS (
    SELECT
        z.zone AS pickup_zone,
        z.borough AS pickup_borough,
        COUNT(*) AS trip_count,
        AVG(f.tip_amount / NULLIF(f.fare_amount, 0)) AS avg_tip_pct
    FROM gold.fct_trips f
    JOIN gold.dim_zone z
        ON f.pickup_zone_id = z.zone_id
    WHERE f.fare_amount > 0
    GROUP BY z.zone, z.borough
)
SELECT
    pickup_zone,
    pickup_borough,
    trip_count,
    ROUND(avg_tip_pct::numeric, 4) AS avg_tip_pct
FROM zone_stats
WHERE trip_count >= 1000
ORDER BY avg_tip_pct DESC
LIMIT 10"""
show_sql(sql)
df_13 = run_sql(sql)
display(df_13)
basic_interpretation(df_13, value_col='avg_tip_pct', label='Top 10 zonas por tip % (pickup) con mínimo N viajes.')


### Pregunta 14
**Comparación cash vs card: viajes, ingreso total, tip %.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(14, 'Comparación cash vs card: viajes, ingreso total, tip %.', 'gold.fct_trips, gold.dim_payment_type')
sql = """SELECT
    CASE
        WHEN LOWER(pt.payment_type_desc) LIKE '%cash%' THEN 'cash'
        WHEN LOWER(pt.payment_type_desc) LIKE '%card%' OR LOWER(pt.payment_type_desc) LIKE '%credit%' THEN 'card'
        ELSE 'other'
    END AS payment_group,
    COUNT(*) AS trip_count,
    ROUND(SUM(f.total_amount)::numeric, 2) AS total_revenue,
    ROUND(AVG(f.tip_amount / NULLIF(f.fare_amount, 0))::numeric, 4) AS avg_tip_pct
FROM gold.fct_trips f
LEFT JOIN gold.dim_payment_type pt
    ON f.payment_type_key = pt.payment_type_key
GROUP BY payment_group
ORDER BY trip_count DESC"""
show_sql(sql)
df_14 = run_sql(sql)
display(df_14)
basic_interpretation(df_14, value_col='trip_count', label='Comparación cash vs card: viajes, ingreso total, tip %.')


### Pregunta 15
**Duración promedio (min) por mes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(15, 'Duración promedio (min) por mes.', 'gold.fct_trips, gold.dim_date')
sql = """SELECT
    d.year,
    d.month,
    ROUND(AVG(f.trip_duration_minutes)::numeric, 2) AS avg_duration_minutes
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month"""
show_sql(sql)
df_15 = run_sql(sql)
display(df_15)
basic_interpretation(df_15, value_col='avg_duration_minutes', label='Duración promedio (min) por mes.')


### Pregunta 16
**Distancia promedio por mes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(16, 'Distancia promedio por mes.', 'gold.fct_trips, gold.dim_date')
sql = """SELECT
    d.year,
    d.month,
    ROUND(AVG(f.trip_distance)::numeric, 2) AS avg_trip_distance
FROM gold.fct_trips f
JOIN gold.dim_date d
    ON f.pickup_date_key = d.date_key
GROUP BY d.year, d.month
ORDER BY d.year, d.month"""
show_sql(sql)
df_16 = run_sql(sql)
display(df_16)
basic_interpretation(df_16, value_col='avg_trip_distance', label='Distancia promedio por mes.')


### Pregunta 17
**Velocidad promedio (mph) por borough y franja horaria.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(17, 'Velocidad promedio (mph) por borough y franja horaria.', 'gold.fct_trips, gold.dim_zone')
sql = """WITH speed_base AS (
    SELECT
        z.borough,
        CASE
            WHEN EXTRACT(HOUR FROM f.pickup_datetime) BETWEEN 6 AND 11 THEN 'morning'
            WHEN EXTRACT(HOUR FROM f.pickup_datetime) BETWEEN 12 AND 17 THEN 'afternoon'
            WHEN EXTRACT(HOUR FROM f.pickup_datetime) BETWEEN 18 AND 22 THEN 'evening'
            ELSE 'night'
        END AS time_band,
        CASE
            WHEN f.trip_duration_minutes > 0 THEN f.trip_distance / (f.trip_duration_minutes / 60.0)
            ELSE NULL
        END AS trip_mph
    FROM gold.fct_trips f
    JOIN gold.dim_zone z
        ON f.pickup_zone_id = z.zone_id
)
SELECT
    borough,
    time_band,
    ROUND(AVG(trip_mph)::numeric, 2) AS avg_mph
FROM speed_base
WHERE trip_mph IS NOT NULL
GROUP BY borough, time_band
ORDER BY borough, time_band"""
show_sql(sql)
df_17 = run_sql(sql)
display(df_17)
basic_interpretation(df_17, value_col='avg_mph', label='Velocidad promedio (mph) por borough y franja horaria.')


### Pregunta 18
**Percentiles p50 y p90 de duración por borough.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(18, 'Percentiles p50 y p90 de duración por borough.', 'gold.fct_trips, gold.dim_zone')
sql = """SELECT
    z.borough,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY f.trip_duration_minutes)::numeric, 2) AS p50_duration_minutes,
    ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes)::numeric, 2) AS p90_duration_minutes
FROM gold.fct_trips f
JOIN gold.dim_zone z
    ON f.pickup_zone_id = z.zone_id
GROUP BY z.borough
ORDER BY p90_duration_minutes DESC"""
show_sql(sql)
df_18 = run_sql(sql)
display(df_18)
basic_interpretation(df_18, value_col='p90_duration_minutes', label='Percentiles p50 y p90 de duración por borough.')


### Pregunta 19
**Top 10 zonas (pickup) por p90 de duración.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(19, 'Top 10 zonas (pickup) por p90 de duración.', 'gold.fct_trips, gold.dim_zone')
sql = """SELECT
    z.zone AS pickup_zone,
    z.borough AS pickup_borough,
    ROUND(PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY f.trip_duration_minutes)::numeric, 2) AS p90_duration_minutes
FROM gold.fct_trips f
JOIN gold.dim_zone z
    ON f.pickup_zone_id = z.zone_id
GROUP BY z.zone, z.borough
ORDER BY p90_duration_minutes DESC
LIMIT 10"""
show_sql(sql)
df_19 = run_sql(sql)
display(df_19)
basic_interpretation(df_19, value_col='p90_duration_minutes', label='Top 10 zonas (pickup) por p90 de duración.')


### Pregunta 20
**Top 10 rutas borough→borough (pickup borough to dropoff borough) por número de viajes.**

Esta pregunta está solicitada en la sección de 20 preguntas de negocio del PSet y debe resolverse usando únicamente tablas `gold.*`. fileciteturn4file0


In [ ]:

show_question(20, 'Top 10 rutas borough→borough (pickup borough to dropoff borough) por número de viajes.', 'gold.fct_trips, gold.dim_zone')
sql = """SELECT
    pu.borough AS pickup_borough,
    do_.borough AS dropoff_borough,
    COUNT(*) AS trip_count
FROM gold.fct_trips f
JOIN gold.dim_zone pu
    ON f.pickup_zone_id = pu.zone_id
JOIN gold.dim_zone do_
    ON f.dropoff_zone_id = do_.zone_id
GROUP BY pu.borough, do_.borough
ORDER BY trip_count DESC
LIMIT 10"""
show_sql(sql)
df_20 = run_sql(sql)
display(df_20)
basic_interpretation(df_20, value_col='trip_count', label='Top 10 rutas borough→borough (pickup borough to dropoff borough) por número de viajes.')


## Entrega
Antes de subir tu repositorio:

1. Ejecuta **Run All** en este notebook.
2. Verifica que todas las consultas lean solo de `gold.*`.
3. Añade debajo de cada resultado una interpretación final de **1–3 líneas** con tus hallazgos.
4. Guarda el notebook con outputs visibles para evidencia de entrega.

## Nota
No pude ejecutar estas consultas contra tu base real desde aquí, así que el notebook queda **completo y listo para correr**, pero los resultados e interpretaciones finales dependerán de tus datos cargados en PostgreSQL.
